# Heatmap — Zwerfafval (grof, fijn) detecties
2026-09-25
Weiyi Ding

Input: `counts_xx.gpkg` generated by `aggregate_detections.ipynb`

Used columns from gpkg:
- `Zwerfafval (grof)` / `Zwerfafval (fijn)` — raw detections per image
- `section_mean_grof` / `section_mean_fijn`
- `rolling_avg_grof` / `rolling_avg_fijn`
- `straat_naam` — straatnaam
- `selected` — 1 selected per 5m
- `timestamp_utc`, `geometry`

In [ ]:
import os
import base64
import json

import geopandas as gpd
import numpy as np
import folium
from folium import CircleMarker
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import LinearSegmentedColormap

In [ ]:
# paths

date = "260824"  # change to switch dataset

GPKG_PATH    = f"/home/ding001/Zwerfafval-Detectie/datasets/experiments/zwerfafval/counts_{date}.gpkg"
IMAGE_FOLDER = f"/home/ding001/Zwerfafval-Detectie/datasets/experiments/zwerfafval/annotatieproject/inwinning_{date}_images"
OUTPUT_PATH  = f"/home/ding001/Zwerfafval-Detectie/datasets/experiments/zwerfafval/heatmap_{date}.html"

GROF_THRESHOLD = 5   # triangles + image shown for grof > this value

# Which column to use for dot color intensity:
# Options: 'Zwerfafval (grof)'  — raw count per image
#          'section_mean_grof'  — mean over the road section
#          'rolling_avg_grof'   — rolling average over neighbouring sections
COLOR_COLUMN = "Zwerfafval (grof)"

In [ ]:
#Load gpkg
gdf = gpd.read_file(GPKG_PATH, layer=0)
print(f"Total rows: {len(gdf)}")
print(f"Selected=True: {gdf['selected'].sum()}")
print(f"Columns: {list(gdf.columns)}")

# Work only with selected points (1 per 5m section)
gdf = gdf[gdf["selected"] == True].copy()
print(f"\nAfter selected=True filter: {len(gdf)} rows")

# NOTE: NO filter on total > 0.
# 0-detection images are valid data points


gdf["total"] = gdf["Zwerfafval (fijn)"] + gdf["Zwerfafval (grof)"]

# Convert to WGS84 for Folium
gdf_wgs = gdf.to_crs("EPSG:4326")

gdf_wgs.head()

In [ ]:
# image index
def build_image_index(folder):
    index = {}
    if not os.path.isdir(folder):
        print(f"  ⚠ Folder not found: {folder}")
        return index
    for fname in os.listdir(folder):
        if fname.lower().endswith((".jpg", ".jpeg", ".png")):
            index[os.path.splitext(fname)[0]] = os.path.join(folder, fname)
    return index

image_index = build_image_index(IMAGE_FOLDER)
print(f"Found {len(image_index)} images")


def get_image_b64(file_name):
    key  = os.path.splitext(os.path.basename(str(file_name)))[0]
    path = image_index.get(key)
    if path and os.path.isfile(path):
        with open(path, "rb") as f:
            data = base64.b64encode(f.read()).decode("utf-8")
        mime = "jpeg" if path.lower().endswith((".jpg", ".jpeg")) else "png"
        return f"data:image/{mime};base64,{data}"
    return None

In [ ]:
# Generate color map, scale (green → orange → red) for both grof and fijn counts
# color is based on the COLOR_COLUMN for grof, and "Zwerfafval (fijn)" for fijn

grof_cmap = LinearSegmentedColormap.from_list("grof", ["#2ecc71", "#e67e22", "#c0392b"])
fijn_cmap = LinearSegmentedColormap.from_list("fijn", ["#2ecc71", "#e67e22", "#c0392b"])

def make_to_hex(series, cmap):
    """Linear color scale: maps series values to hex colors."""
    non_zero = series[series > 0]
    vmin = non_zero.min() if len(non_zero) > 0 else 0
    vmax = series.max()
    norm = mcolors.Normalize(vmin=vmin, vmax=max(vmax, 1))
    def to_hex(val):
        if val == 0:
            return "#cccccc"  # grey for zero-detection points
        return mcolors.to_hex(cmap(norm(val)))
    return to_hex, round(float(vmin), 1), round((float(vmin) + float(vmax)) / 2, 1), round(float(vmax), 1)

to_hex_grof, grof_min, grof_mid, grof_max = make_to_hex(gdf_wgs[COLOR_COLUMN], grof_cmap)
to_hex_fijn, fijn_min, fijn_mid, fijn_max = make_to_hex(gdf_wgs["Zwerfafval (fijn)"], fijn_cmap)

print(f"Grof scale ({COLOR_COLUMN}): {grof_min} → {grof_mid} → {grof_max}")
print(f"Fijn scale: {fijn_min} → {fijn_mid} → {fijn_max}")

In [ ]:
## Build map ##

# Zwerfafval (grof) — raw detection count for this individual image
# section_mean_grof — mean count over the 5m road section around this point
# rolling_avg_grof — rolling average over neighbouring sections (~25m)
# straat_naam — street names joined from Amsterdam geodata in aggregate_detections.ipynb (no external API needed here)
# iterrows() is used so all columns are accessible by name

center = [gdf_wgs.geometry.y.mean(), gdf_wgs.geometry.x.mean()]

m = folium.Map(location=center, zoom_start=13, tiles=None)
folium.TileLayer(
    tiles="https://t1.data.amsterdam.nl/topo_wm/{z}/{x}/{y}.png",
    attr="data.amsterdam.nl",
    name="Amsterdam Topo",
).add_to(m)

layer_grof = folium.FeatureGroup(name="Zwerfafval (grof)", show=True)
layer_fijn = folium.FeatureGroup(name="Zwerfafval (fijn)", show=True)

missing_img = 0


def triangle_icon(color):
    """Small upward triangle for hotspot points (grof > GROF_THRESHOLD)."""
    return folium.DivIcon(
        html=(
            f'<div style="width:0;height:0;'
            f'border-left:6px solid transparent;'
            f'border-right:6px solid transparent;'
            f'border-bottom:11px solid {color};'
            f'filter:drop-shadow(0 0 1px rgba(0,0,0,0.6));'
            f'margin-top:-5px;margin-left:-6px"></div>'
        ),
        icon_size=(12, 11),
        icon_anchor=(6, 5),
    )


def make_tooltip(row, grof, fijn, color, img_html=""):
    """Tooltip with street name (from gpkg), datetime, and counts."""
    # straat_naam is in the gpkg, no API needed
    straat   = row.get("straat_naam", "Onbekend") or "Onbekend"
    ts       = row.get("timestamp_utc")
    dt_date  = str(ts)[:10]   if ts is not None and str(ts) != "NaT" else "Onbekend"
    dt_time  = str(ts)[11:16] if ts is not None and str(ts) != "NaT" else "Onbekend"
    sec_grof = row.get("section_mean_grof")
    rol_grof = row.get("rolling_avg_grof")
    badge = (
        f'<span style="display:inline-block;background:{color};color:white;'
        f'border-radius:4px;padding:1px 7px;font-size:10px;font-weight:600;'
        f'margin-bottom:6px">Grof {grof}</span>'
    )
    extra_rows = ""
    if sec_grof is not None and not np.isnan(float(sec_grof)):
        extra_rows += f'<tr><td style="color:#888;white-space:nowrap">Vak gem.</td><td style="color:#444">{float(sec_grof):.2f}</td></tr>'
    if rol_grof is not None and not np.isnan(float(rol_grof)):
        extra_rows += f'<tr><td style="color:#888;white-space:nowrap">Rolling avg</td><td style="color:#444">{float(rol_grof):.2f}</td></tr>'
    return folium.Tooltip(
        f"""<div style="font-family:system-ui;font-size:12px;max-width:270px">
          {img_html}
          {badge}
          <table style="width:100%;border-collapse:collapse;line-height:1.8">
            <tr><td style="color:#888;padding-right:8px;white-space:nowrap">📍 Straat</td>
                <td style="font-weight:600;color:#111">{straat}</td></tr>
            <tr><td style="color:#888;white-space:nowrap">📅 Datum</td>
                <td style="color:#444">{dt_date}</td></tr>
            <tr><td style="color:#888;white-space:nowrap">🕐 Tijd</td>
                <td style="color:#444">{dt_time}</td></tr>
            <tr><td style="color:#888;white-space:nowrap">📦 Grof</td>
                <td style="color:#444">{grof}</td></tr>
            <tr><td style="color:#888;white-space:nowrap">📦 Fijn</td>
                <td style="color:#444">{fijn}</td></tr>
            {extra_rows}
          </table>
        </div>""",
        sticky=True,
    )


for _, row in gdf_wgs.iterrows():
    # Access all columns by name, no index needed
    lat   = row.geometry.y
    lon   = row.geometry.x
    grof  = int(row["Zwerfafval (grof)"])   # raw count for this image
    fijn  = int(row["Zwerfafval (fijn)"])    # raw count for this image
    fname = str(row["file_name"]) if "file_name" in row.index else ""
    color_val = row[COLOR_COLUMN] if COLOR_COLUMN in row.index else grof

    # Grof layer
    color_grof = to_hex_grof(float(color_val))

    if grof > GROF_THRESHOLD:
        img_b64 = get_image_b64(fname)
        if img_b64:
            img_html = (
                f'<img src="{img_b64}" style="width:260px;height:170px;'
                f'object-fit:cover;border-radius:5px;display:block;margin-bottom:8px">'
            )
        else:
            missing_img += 1
            img_html = (
                '<div style="width:260px;height:40px;background:#f0f0ee;border-radius:5px;'
                'display:flex;align-items:center;justify-content:center;'
                'color:#bbb;font-size:11px;margin-bottom:8px">Geen afbeelding</div>'
            )
        folium.Marker(
            location=[lat, lon],
            icon=triangle_icon(color_grof),
            tooltip=make_tooltip(row, grof, fijn, color_grof, img_html),
        ).add_to(layer_grof)
    else:
        CircleMarker(
            location=[lat, lon],
            radius=4,
            color="#000000", fill=True, fill_color=color_grof,
            fill_opacity=0.75, weight=1,
            tooltip=make_tooltip(row, grof, fijn, color_grof),
        ).add_to(layer_grof)

    # Fijn layer
    color_fijn = to_hex_fijn(float(row["Zwerfafval (fijn)"]))
    CircleMarker(
        location=[lat, lon],
        radius=4,
        color=color_fijn, fill=True, fill_color=color_fijn,
        fill_opacity=0.6, weight=0.5,
        tooltip=make_tooltip(row, grof, fijn, color_fijn),
    ).add_to(layer_fijn)

layer_grof.add_to(m)
layer_fijn.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)

print(f"Triangles (grof > {GROF_THRESHOLD}): {(gdf_wgs['Zwerfafval (grof)'] > GROF_THRESHOLD).sum()}")
print(f"Dots: {(gdf_wgs['Zwerfafval (grof)'] <= GROF_THRESHOLD).sum()}")
print(f"Missing images: {missing_img}")

In [ ]:
#Legend
legend_html = f"""
<div style="position:fixed;bottom:40px;left:20px;z-index:9999;
    background:white;border-radius:8px;padding:12px 16px;
    box-shadow:0 1px 6px rgba(0,0,0,.2);font-family:system-ui;font-size:12px">
  <div style="font-weight:600;margin-bottom:10px;color:#111">Detectiedichtheid</div>

  <div style="font-size:11px;font-weight:600;color:#444;margin-bottom:4px">Grof ({COLOR_COLUMN})</div>
  <div style="width:150px;height:12px;border-radius:3px;
    background:linear-gradient(to right,#2ecc71,#e67e22,#c0392b)"></div>
  <div style="display:flex;justify-content:space-between;width:150px;margin-top:3px;color:#666;font-size:10px">
    <span>{grof_min}</span><span>{grof_mid}</span><span>{grof_max}</span>
  </div>

  <div style="font-size:11px;font-weight:600;color:#444;margin:12px 0 4px">Fijn</div>
  <div style="width:150px;height:12px;border-radius:3px;
    background:linear-gradient(to right,#2ecc71,#e67e22,#c0392b)"></div>
  <div style="display:flex;justify-content:space-between;width:150px;margin-top:3px;color:#666;font-size:10px">
    <span>{fijn_min}</span><span>{fijn_mid}</span><span>{fijn_max}</span>
  </div>

  <div style="margin-top:10px;font-size:10px;color:#aaa">
    ▲ grof &gt; {GROF_THRESHOLD} + foto &nbsp;|&nbsp; ● overige punten &nbsp;|&nbsp; ◯ grijs = 0 detecties
  </div>
</div>"""

m.get_root().html.add_child(folium.Element(legend_html))

In [ ]:
# Save output
m.save(OUTPUT_PATH)
print(f"Saved → {OUTPUT_PATH}  ({os.path.getsize(OUTPUT_PATH)/1024/1024:.1f} MB)")
print(f"Points plotted: {len(gdf_wgs)}")

In [ ]:
# Ranking most grof counts on streets for statstics
top20 = (
    gdf_wgs.groupby("straat_naam")["Zwerfafval (grof)"]
    .agg(
        **{
            "Grof (totaal)":     "sum",
            "Aantal punten":     "count",
            "Gem. grof per foto": "mean",
        }
    )
    .sort_values("Grof (totaal)", ascending=False)
    .head(20)
    .reset_index()
    .rename(columns={"straat_naam": "Straat"})
)

top20["Gem. grof per foto"] = top20["Gem. grof per foto"].round(1)
top20.index += 1  # rank starts at 1

top20